
# H19a: One-step local Hessian probe for partial SV equalization

**Pair identity.** This notebook is the analysis/reporting counterpart to `run_experiment.py` in the same experiment directory. The script now owns the experiment logic and returns structured results; the notebook imports those results and presents them transparently.

**Truthful scope.** This is a **toy 3-layer deep linear 4×4 study** with a tractable 48×48 finite-difference Hessian. It probes **local curvature before and after one transformed step** taken from a warmup point. It is **not** a trajectory study, convergence study, or general proof that partial SV equalization creates spurious saddles.

**Question under test.** After 50 momentum-SGD warmup steps, do intermediate per-layer equalization depths `k ∈ {1,2,3}` increase local negative Hessian curvature more than the extremes `k=0` and `k=4` after one transformed step?

**Primary metrics.**
- `baseline_neg`, `after_neg`, and `delta_neg = after_neg - baseline_neg`
- `baseline_min`, `after_min`, and `delta_min = after_min - baseline_min`

The notebook deliberately uses the script's returned results instead of reimplementing the experiment core.



---
## 1. Notebook-safe imports and explicit path resolution

This section avoids notebook dependence on `__file__`. It searches for the repository root explicitly, then imports `run_experiment.py` by path via `importlib`.


In [ ]:

import importlib.util
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
pd.set_option('display.precision', 6)

TARGET_RELATIVE = Path('experiments/Tier_1_Core_Mechanism_Tests/H19a_PARTIAL_SV_SPURIOUS_SADDLES/run_experiment.py')
KNOWN_REPO_ROOT = Path('/home/secemp9/Muon_as_RG_Gauge_Fixing')


def locate_repo_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if KNOWN_REPO_ROOT.exists():
        candidates.append(KNOWN_REPO_ROOT.resolve())

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / TARGET_RELATIVE).exists():
            return candidate

    raise FileNotFoundError(
        f'Could not locate repo root containing {TARGET_RELATIVE}. '
        'Launch the notebook inside the repo or update KNOWN_REPO_ROOT.'
    )


def load_module_from_path(module_name, module_path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(f'Could not create loader for {module_path}')
    spec.loader.exec_module(module)
    return module


REPO_ROOT = locate_repo_root()
SCRIPT_PATH = REPO_ROOT / TARGET_RELATIVE
h19a = load_module_from_path('h19a_partial_sv_spurious_saddles', SCRIPT_PATH)

print(f'Repo root:   {REPO_ROOT}')
print(f'Script path: {SCRIPT_PATH}')
print('Notebook-safe import succeeded.')



---
## 2. Execute the script-owned experiment once

The notebook calls the script's `run_experiment()` function directly, then reuses its structured outputs. This keeps the pair aligned by construction.


In [ ]:

t0 = time.perf_counter()
results = h19a.run_experiment(verbose=False)
runtime_sec = time.perf_counter() - t0

h19a.print_results_summary(results)
print(f'Notebook-side wall-clock runtime: {runtime_sec:.3f} s')



---
## 3. Reproducibility, configuration, and runtime

This is the minimum configuration record needed to understand what was actually run. The cost numbers below are based on the current finite-difference Hessian implementation in the script.


In [ ]:

config = results['config']

config_rows = [
    ('experiment_id', results['experiment_id']),
    ('title', results['title']),
    ('question', results['question']),
    ('repo_root', str(REPO_ROOT)),
    ('script_path', str(SCRIPT_PATH)),
    ('runtime_sec', runtime_sec),
    ('seeds', results['seeds']),
    ('k_per_layer', config['k_per_layer']),
    ('dim', config['dim']),
    ('n_layers', config['n_layers']),
    ('n_params', config['n_params']),
    ('n_samples', config['n_samples']),
    ('data_std', config['data_std']),
    ('init_std', config['init_std']),
    ('warmup_steps', config['warmup_steps']),
    ('momentum', config['momentum']),
    ('lr', config['lr']),
    ('fd_eps', config['fd_eps']),
    ('neg_eig_threshold', config['neg_eig_threshold']),
    ('hessians_per_seed', config['hessians_per_seed']),
    ('grad_evals_per_hessian', config['grad_evals_per_hessian']),
    ('hessian_grad_evals_per_seed', config['hessian_grad_evals_per_seed']),
    ('minimum_total_grad_evals_per_seed', config['minimum_total_grad_evals_per_seed']),
    ('scope', config['scope']),
]
config_df = pd.DataFrame(config_rows, columns=['item', 'value'])
display(config_df)

print('k=1 note:')
print('  ' + results['notes']['k1_behavior'])
print('Positive-k step-norm note:')
print('  ' + results['notes']['positive_k_step_norm_note'])
print('Unused helper note:')
print('  ' + results['notes']['unused_helper'])



---
## 4. Aggregate per-k results table

The primary table below is the most compact serious summary of the experiment. It includes baseline curvature, post-step curvature, and explicit deltas. Because the baseline negative-eigenvalue count is already large, `delta_neg` is the more informative headline quantity than `after_neg` alone.


In [ ]:

seed_df = pd.DataFrame(results['seed_k_records']).sort_values(['seed', 'k']).reset_index(drop=True)
agg_df = pd.DataFrame(results['aggregate_by_k']).sort_values('k').reset_index(drop=True)

agg_view = agg_df[[
    'k',
    'mean_baseline_neg', 'std_baseline_neg',
    'mean_after_neg', 'std_after_neg',
    'mean_delta_neg', 'std_delta_neg',
    'mean_baseline_min',
    'mean_after_min',
    'mean_delta_min', 'std_delta_min',
    'mean_step_norm', 'std_step_norm',
    'mean_new_loss', 'std_new_loss',
    'mean_after_symmetry_residual',
    'k_behavior',
]].copy()

display(agg_view.round(6))



### Seed-level table

This seed-resolved table is useful for checking whether any aggregate pattern is driven by a single seed.


In [ ]:

seed_view = seed_df[[
    'seed', 'k',
    'baseline_neg', 'after_neg', 'delta_neg',
    'baseline_min', 'after_min', 'delta_min',
    'step_norm', 'new_loss',
    'baseline_symmetry_residual', 'after_symmetry_residual',
]].copy()

display(seed_view.round(6))



---
## 5. Curvature plots with seed visibility

The first plot emphasizes the primary metric `delta_neg`; the second figure shows minimum-eigenvalue behavior. Seed-level trajectories are shown faintly underneath the aggregate mean ± standard deviation.


In [ ]:

fig, ax = plt.subplots(figsize=(7, 4))
for seed, group in seed_df.groupby('seed'):
    group = group.sort_values('k')
    ax.plot(group['k'], group['delta_neg'], color='0.75', marker='o', linewidth=1, alpha=0.8)

ax.errorbar(
    agg_df['k'],
    agg_df['mean_delta_neg'],
    yerr=agg_df['std_delta_neg'],
    color='C0',
    marker='o',
    linewidth=2,
    capsize=4,
    label='mean ± std',
)
ax.axhline(0.0, color='black', linestyle='--', linewidth=1)
ax.set_title('Change in negative Hessian eigenvalue count after one step')
ax.set_xlabel('per-layer equalization depth k')
ax.set_ylabel('delta_neg = after_neg - baseline_neg')
ax.set_xticks(agg_df['k'])
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for seed, group in seed_df.groupby('seed'):
    group = group.sort_values('k')
    axes[0].plot(group['k'], group['after_min'], color='0.75', marker='o', linewidth=1, alpha=0.8)
    axes[1].plot(group['k'], group['delta_min'], color='0.75', marker='o', linewidth=1, alpha=0.8)

axes[0].errorbar(
    agg_df['k'],
    agg_df['mean_after_min'],
    yerr=agg_df['std_after_min'],
    color='C1',
    marker='o',
    linewidth=2,
    capsize=4,
)
axes[0].set_title('Minimum Hessian eigenvalue after one step')
axes[0].set_xlabel('per-layer equalization depth k')
axes[0].set_ylabel('after_min')
axes[0].set_xticks(agg_df['k'])
axes[0].grid(alpha=0.25)

axes[1].errorbar(
    agg_df['k'],
    agg_df['mean_delta_min'],
    yerr=agg_df['std_delta_min'],
    color='C2',
    marker='o',
    linewidth=2,
    capsize=4,
)
axes[1].axhline(0.0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Change in minimum Hessian eigenvalue after one step')
axes[1].set_xlabel('per-layer equalization depth k')
axes[1].set_ylabel('delta_min = after_min - baseline_min')
axes[1].set_xticks(agg_df['k'])
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()



---
## 6. Interpretation of the current default run

This cell turns the structured outputs into explicit statements about what the current experiment did and did not show.


In [ ]:

tests = results['tests']
step_norm_summary = seed_df.groupby('k')['step_norm'].agg(['mean', 'std']).reset_index()
display(step_norm_summary.round(6))

print('Headline checks:')
print(f"  Intermediate k increases mean after-neg count: {tests['intermediate_k_increases_mean_after_neg']}")
print(f"  Intermediate k increases mean delta_neg: {tests['intermediate_k_increases_mean_delta_neg']}")
print(f"  All observed delta_neg values are zero: {tests['all_delta_neg_zero']}")
print(f"  Positive-k step norms match by construction: {tests['positive_k_step_norms_match']}")
print(f"  k=1 identical to k=0 under the current implementation: {tests['k1_is_identical_to_k0']}")
print()
print('Interpretive notes:')
print('  - Under the current implementation, k=1 is DISTINCT from k=0.')
print("    It keeps each layer's singular directions and relative singular-value ratios, but rescales the whole layer update to Frobenius norm sqrt(d).")
print('  - Therefore k=1 is best viewed as a norm-normalized raw-gradient variant, not as literal no-op equivalence with SGD.')
print('  - All k>0 updates are step-norm matched with each other; k=0 is not step-norm matched to them.')
if tests['all_delta_neg_zero']:
    print('  - In the current default run, every seed and every k gives delta_neg = 0.')
if not tests['intermediate_k_increases_mean_after_neg']:
    print('  - So the present run does NOT show an intermediate-k increase in negative-eigenvalue count.')



---
## 7. Calibrated conclusion

The conclusion below is intentionally tied to the observed result rather than the original stronger title-level claim.


In [ ]:

baseline_mean = float(agg_df.loc[agg_df['k'] == 0, 'mean_baseline_neg'].iloc[0])
after0 = float(agg_df.loc[agg_df['k'] == 0, 'mean_after_neg'].iloc[0])
after4 = float(agg_df.loc[agg_df['k'] == 4, 'mean_after_neg'].iloc[0])

delta_sentence = (
    'Every observed default-run value of delta_neg is zero across all seeds and all k values.'
    if tests['all_delta_neg_zero']
    else 'The default run shows nonzero delta_neg values for at least some seed/k pairs.'
)

headline_sentence = (
    'The current run therefore does not support the claim that intermediate k creates more local saddle directions than both extremes in this probe.'
    if not tests['intermediate_k_increases_mean_after_neg']
    else 'The current run shows an intermediate-k increase in local saddle directions under this probe.'
)

conclusion_md = f"""
### Final take-away

Under the default H19a configuration, this notebook reproduces the script's **one-step local Hessian probe** on the same toy deep linear network.

- The warmup baseline already has about **{baseline_mean:.1f} negative Hessian eigenvalues on average**.
- Mean post-step negative-eigenvalue counts at the two extremes are **k=0: {after0:.1f}** and **k=4: {after4:.1f}**.
- {delta_sentence}
- {headline_sentence}
- Under the current implementation, **k=1 is not identical to k=0**; it differs mainly by layerwise Frobenius normalization, so it should not be narrated as a separate partially equalized geometry effect without that caveat.
- Consequently, this first-pass pair should be read as a **reproducible local-curvature benchmark** with calibrated interpretation, not as a demonstration that partial SV equalization generally creates spurious saddles.
"""

display(Markdown(conclusion_md))
